# Demo 4 — Ba loại data distribution shift

**Slide:** 9–13 · **Thời lượng:** 60–90 giây · **Hình:** `04_shift_types.png`

## Ý chính

Cùng một model logistic, train trên **source**, đánh giá trên **target**:

| Loại | Thay đổi | Giữ nguyên |
|---|---|---|
| Covariate shift | $P(X)$ | $P(Y \mid X)$ |
| Label shift | $P(Y)$ | $P(X \mid Y)$ |
| Concept drift | $P(Y \mid X)$ | $P(X)$ |

> **Câu chốt:** mỗi loại shift cần cách đo và cách ứng phó khác nhau.

## Bước 0 — Thiết lập

In [ ]:
# --- Thiết lập: tìm và import helper từ ../code-demo ---
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

for candidate in [
    Path.cwd() / "code-demo",
    Path.cwd().parent / "code-demo",
    Path.cwd().parent.parent / "code-demo",
]:
    if (candidate / "common.py").exists():
        CODE_DIR = candidate.resolve()
        break
else:
    raise FileNotFoundError("Không tìm thấy code-demo/common.py")

sys.path.insert(0, str(CODE_DIR))

from common import accuracy, predict, save, setup, sigmoid, train_logreg  # noqa: E402

setup()
print("code-demo:", CODE_DIR)
print("Hình sẽ lưu vào:", CODE_DIR / "figures")

## Bước 1 — Định nghĩa 3 kịch bản

- **Covariate:** train chỉ thấy vùng $x$ âm, production chỉ có vùng $x$ dương. Quy luật $y=1$ khi $|x|>1$ không đổi.
- **Label:** tỷ lệ nhãn đổi từ 50% xuống 10%; phân phối $x$ trong từng lớp giữ nguyên.
- **Concept:** cùng $x$, quy luật bị đảo — $y=1$ khi $x>0$ thành $y=1$ khi $x<0$.

In [ ]:
N = 4000


def empirical_p_y_given_x(x, y, bins=18):
    edges = np.linspace(np.quantile(x, 0.01), np.quantile(x, 0.99), bins + 1)
    idx = np.clip(np.digitize(x, edges) - 1, 0, bins - 1)
    centers, probs = [], []
    for b in range(bins):
        mask = idx == b
        if mask.sum() >= 20:
            centers.append(x[mask].mean())
            probs.append(y[mask].mean())
    return np.array(centers), np.array(probs)


def make_covariate(rng):
    def true_p(x):
        return sigmoid(6.0 * (np.abs(x) - 1.0))

    Xs = rng.normal(-1.6, 0.35, N)
    ys = rng.binomial(1, true_p(Xs))
    Xt = rng.normal(1.6, 0.35, N)
    yt = rng.binomial(1, true_p(Xt))
    return Xs, ys, Xt, yt


def make_label(rng):
    def sample(prior):
        y = rng.binomial(1, prior, N)
        X = np.where(y == 1, rng.normal(1.0, 1.0, N), rng.normal(-1.0, 1.0, N))
        return X, y

    Xs, ys = sample(0.5)
    Xt, yt = sample(0.1)
    return Xs, ys, Xt, yt


def make_concept(rng):
    Xs = rng.normal(0, 1, N)
    ys = (Xs > 0).astype(int)
    Xt = rng.normal(0, 1, N)
    yt = (Xt < 0).astype(int)   # quy luật bị đảo
    return Xs, ys, Xt, yt


rng = np.random.default_rng(11)
scenarios = [
    ("Covariate shift\nP(X) đổi, P(Y|X) giữ nguyên", make_covariate(rng)),
    ("Label shift\nP(Y) đổi, P(X|Y) giữ nguyên", make_label(rng)),
    ("Concept drift\nP(Y|X) đổi, P(X) giữ nguyên", make_concept(rng)),
]
print("Đã tạo", len(scenarios), "kịch bản, mỗi kịch bản", N, "mẫu source +", N, "mẫu target.")

## Bước 2 — Train trên source, đánh giá trên target

In [ ]:
print("=== DEMO 4: ba loại shift ===")
print(f"{'loại':>18} | {'acc source':>10} | {'acc target':>10} | {'giảm':>6}")
print("  " + "-" * 54)

results = []
for title, (Xs, ys, Xt, yt) in scenarios:
    w = train_logreg(Xs, ys, lr=0.2, epochs=1200)
    acc_s = accuracy(ys, predict(Xs, w))
    acc_t = accuracy(yt, predict(Xt, w))
    results.append((title, Xs, ys, Xt, yt, w, acc_s, acc_t))
    print(f"{title.splitlines()[0]:>18} | {acc_s:>10.3f} | {acc_t:>10.3f} | {acc_s - acc_t:>+6.3f}")
    print(
        f"   → P(y=1): source={ys.mean():.2f}, target={yt.mean():.2f}; "
        f"model dự đoán positive={predict(Xt, w).mean():.2f}"
    )

print()
print("Label shift thường không làm accuracy sụt mạnh, nhưng làm model dự đoán sai TỶ LỆ nhãn.")
print("Vì vậy cần monitor cả P(Y) và positive rate, không chỉ accuracy.")

## Bước 3 — Vẽ phân phối input và $P(y=1 \mid x)$

- Hàng trên: histogram $P(X)$ của source và target.
- Hàng dưới: xác suất $y=1$ theo $x$ — cho thấy quy luật có đổi hay không.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))

for col, (title, Xs, ys, Xt, yt, w, acc_s, acc_t) in enumerate(results):
    # Hàng trên: phân phối input
    ax = axes[0, col]
    bins = np.linspace(-5.5, 5.5, 45)
    ax.hist(Xs, bins=bins, density=True, alpha=0.55, label="source (train)", color="#1f77b4")
    ax.hist(Xt, bins=bins, density=True, alpha=0.55, label="target (production)", color="#d62728")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x")
    ax.set_ylabel("mật độ")
    ax.legend(fontsize=8)

    # Hàng dưới: xác suất y=1 theo x (empirical)
    ax = axes[1, col]
    xs_c, xs_p = empirical_p_y_given_x(Xs, ys)
    xt_c, xt_p = empirical_p_y_given_x(Xt, yt)
    ax.plot(xs_c, xs_p, "o-", label="source", color="#1f77b4", ms=3)
    ax.plot(xt_c, xt_p, "o-", label="target", color="#d62728", ms=3)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("x")
    ax.set_ylabel("P(y=1 | x)")
    ax.set_title(f"Accuracy: {acc_s:.2f} → {acc_t:.2f}", fontsize=10)
    ax.legend(fontsize=8)

fig.suptitle("Cùng một model, ba kiểu lệch phân phối khác nhau", fontsize=13)

save(fig, "04_shift_types.png")
plt.show()

## Đọc kết quả & ý nghĩa cho slide

| Loại | Accuracy source | Accuracy target | Điều đáng chú ý |
|---|---|---|---|
| Covariate shift | 0.91 | **0.10** | Model extrapolate sai vùng input mới |
| Label shift | 0.85 | 0.84 | Nhưng dự đoán positive **0.24** vs thật **0.11** |
| Concept drift | 1.00 | **0.00** | Quy luật cũ vô dụng |

**Nói trên slide:**

> “Cùng một model. Covariate shift: accuracy 0,91 xuống 0,10 vì model chưa từng thấy vùng input mới. Label shift: accuracy không giảm mạnh, nhưng model dự đoán positive 0,24 trong khi tỷ lệ thật chỉ 0,11. Concept drift: quy luật đảo, accuracy gần 0.”

**Câu chốt:** mỗi loại shift cần cách đo và cách ứng phó khác nhau.

**Dùng cho slide nào?**

- Slide 9 — data distribution shift là gì.
- Slide 10 — covariate shift.
- Slide 11 — label shift.
- Slide 12 — concept drift.
- Slide 13 — feature change & label schema change (dùng bảng so sánh).